# OpenAI Calls: GPT 5.4 (latest model)

In [ ]:
from openai import OpenAI
print("openai import worked")

In [ ]:
import os
import re
import io
import pandas as pd
from getpass import getpass
import time

In [ ]:
api_key = getpass("Paste your OpenAI API key: ")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Set it in your environment or paste it manually.")
else:
  print("ALL GOOD!")

## Data Collection

In [ ]:
# =========================================================
# a) Settings
# =========================================================
file_path = "DJIA.xlsx"
sheet_name = "Sheet1"

start_date = "2018-01-01"
end_date = "2024-06-30"

tickers = [
    "MMM.N","AMZN.OQ","AXP.N","AMGN.OQ","AAPL.OQ","BA.N","CAT.N","CVX.N",
    "CSCO.OQ","KO.N","DOW.N","GS.N","HD.N","HON.OQ","IBM.N","INTC.OQ",
    "JNJ.N","JPM.N","MCD.N","MRK.N","MSFT.OQ","NKE.N","PG.N","CRM.N",
    "TRV.N","UNH.N","VZ.N","V.N","WMT.N","DIS.N"
]

tickers_clean = [t.split(".")[0] for t in tickers]

# =========================================================
# b) Read weekly returns from the Excel
# =========================================================
df = pd.read_excel(file_path, sheet_name=sheet_name)

df["Date"] = pd.to_datetime(df["Date"])
df = df.set_index("Date").sort_index()

weekly_returns = df[tickers].copy()
weekly_returns.columns = tickers_clean

weekly_returns = weekly_returns.loc[start_date:end_date]

if weekly_returns.empty:
    raise ValueError("No data found in the selected date range.")

print("Weekly returns raw shape:", weekly_returns.shape)

# =========================================================
# c) Compute WEEKLY mu and Sigma
# =========================================================
# Weekly mean returns
mu_weekly = weekly_returns.mean()

# Weekly covariance matrix
Sigma_weekly = weekly_returns.cov()

print(mu_weekly.shape)
print(Sigma_weekly.shape)

# =========================================================
# d) Save mu and Sigma
# =========================================================
mu_weekly.to_csv("mu_djia_data.csv")
Sigma_weekly.to_csv("sigma_djia_data.csv")

# =========================================================
# f) Format mu and Sigma for the prompt
# =========================================================
mu_text = "\n".join([f"{ticker}: {value:.4f}" for ticker, value in mu_weekly.items()])
sigma_text = Sigma_weekly.round(4).to_csv()

# =========================================================
# g) Weekly summary stats
# =========================================================
summary_stats = pd.DataFrame({
    "mean": weekly_returns.mean(),
    "std": weekly_returns.std(),
    "min": weekly_returns.min(),
    "max": weekly_returns.max(),
    "skew": weekly_returns.skew(),
    "kurtosis_excess": weekly_returns.kurt()
})

print("\nWeekly summary stats:")
print(summary_stats)

## Prompts

In [ ]:
# API key
client = OpenAI(api_key=api_key)

ticker_list_text = ", ".join(tickers_clean)
n_assets = len(tickers)

# PROMPTS
prompts = {
    "L0": f"""You are advising a retail investor.

Construct a portfolio using these {n_assets} stocks: {ticker_list_text}.

The investor is risk-averse and wants a diversified portfolio.

Constraints:
- No short selling
- Weights must be non-negative
- Weights must sum exactly to 1

Return the output in exactly the following format:

PORTFOLIO_CSV
ticker,weight
[{n_assets} rows]

COMMENT
Provide a short explanation (maximum 3 sentences) of how the portfolio was
constructed.

Do not use markdown.
Do not include any additional text.
""",

    "L1": f"""You are advising a retail investor.

Construct a portfolio over these stocks: {ticker_list_text}. The investor is risk-averse and wants a diversified portfolio that balances
expected return and risk.

Take into account:
- higher expected returns are desirable
- lower risk (volatility and correlations) is preferable

Constraints:
- No short selling
- Weights must be non-negative
- Weights must sum exactly to 1

You are given the following data:

Expected returns (mu):
{mu_text}

Covariance matrix (Sigma):
{sigma_text}

Return the output in exactly the following format:

PORTFOLIO_CSV
ticker,weight
[{n_assets} rows]

COMMENT
Provide a short explanation (maximum 3 sentences) of how the portfolio was
constructed.

Do not use markdown.
Do not include any additional text.
""",

    "L2": f"""You are advising a retail investor.
Construct a portfolio over {ticker_list_text} using a mean-variance optimization framework. Use a fixed parameter lambda = 0.5.

Use the expected returns and covariance matrix to balance return against risk, while respecting the constraints below:
- No short selling
- Weights must be non-negative
- Weights must sum exactly to 1

The investor is risk-averse and wants a diversified portfolio.

You are given:

Expected returns (mu):
{mu_text}

Covariance matrix (Sigma):
{sigma_text}

Return the output in exactly the following format:

PORTFOLIO_CSV
ticker,weight
[{n_assets} rows]

COMMENT
Provide a short explanation (maximum 3 sentences) of how the portfolio was constructed.

Do not use markdown.
Do not include any additional text.
""",

    "L3": f"""You are advising a retail investor.

Construct a portfolio over {ticker_list_text} using constrained mean-variance optimization. Use a fixed parameter lambda = 0.5.

Choose portfolio weights w to solve:

max_w    mu^T w - (lambda/2) * w^T Sigma w

subject to:
- w_i >= 0 for all i
- sum_i w_i = 1

This is a constrained quadratic optimization problem. If exact closed form
matrix inversion is not feasible because of no-short-selling constraints,
reason in terms of numerical optimization (e.g. quadratic programming).

The investor is risk-averse and wants a diversified portfolio.

You are given:

Expected returns (mu):
{mu_text}

Covariance matrix (Sigma):
{sigma_text}

Steps:
1. Use the expected returns and covariance matrix
2. Apply mean-variance logic
3. Ensure that all weights are non-negative and sum exactly to 1.
4. Return the final portfolio weights

Return the output in exactly the following format:

PORTFOLIO_CSV
ticker,weight
[{n_assets} rows]

COMMENT
Provide a short explanation (maximum 3 sentences) of how the portfolio was
constructed.

Do not use markdown.
Do not include any additional text.
"""
}

results = []

## Loop through the Levels

In [ ]:
output_file = "results_all_gpt5.4.csv"
comments_file = "comments_all_gpt5.4.csv"

# =========================================================
# 1) LOAD EXISTING RESULTS (if any)
# =========================================================
if os.path.exists(output_file):
    existing_df = pd.read_csv(output_file)
    start_rep = existing_df["rep"].max() + 1
    print(f"Continuing from rep {start_rep}")
else:
    existing_df = pd.DataFrame()
    start_rep = 0
    print("Starting from rep 0")

if os.path.exists(comments_file):
    existing_comments = pd.read_csv(comments_file)
else:
    existing_comments = pd.DataFrame()

# =========================================================
# 2) HOW MANY NEW REPS YOU WANT NOW
# =========================================================
n_new_rep = 78

# =========================================================
# 3) RUN NEW BATCH
# =========================================================
new_results = []
new_comments = []

for rep in range(start_rep, start_rep + n_new_rep):
    print(f"\n--- Rep {rep} ---")

    for level, prompt in prompts.items():
        response = client.responses.create(
            model="gpt-5.4",
            input=prompt,
            temperature=0.2
        )
        time.sleep(3)

        output = response.output_text

        match = re.search(r"PORTFOLIO_CSV\s*(.*?)\s*COMMENT", output, re.DOTALL)

        #extract the comment
        comment_match = re.search(r"COMMENT\s*(.*)", output, re.DOTALL)
        if comment_match:
            comment_text = comment_match.group(1).strip()
        else:
            comment_text = ""

        new_comments.append({
            "level": level,
            "rep": rep,
            "comment": comment_text})

        if match:
            csv_text = match.group(1).strip()

            try:
                df = pd.read_csv(io.StringIO(csv_text))
                df["weight"] = pd.to_numeric(df["weight"], errors="raise")

                df["level"] = level
                df["rep"] = rep

                new_results.append(df)

            except Exception as e:
                print(f"Parsing error in {level}, rep {rep}: {e}")

        else:
            print(f"No CSV found in {level}, rep {rep}")

    # =========================================================
    # 4) SAVE RESULTS
    # =========================================================
    if new_results:
        new_df = pd.concat(new_results, ignore_index=True)
        final_df = pd.concat([existing_df, new_df], ignore_index=True)
        final_df.to_csv(output_file, index=False)

        print(f"Total reps now: {final_df['rep'].nunique()}")

    else:
        print("No new results to save.")

    if new_comments:
        new_comments_df = pd.DataFrame(new_comments)
        final_comments_df = pd.concat([existing_comments, new_comments_df], ignore_index=True)
        final_comments_df.to_csv(comments_file, index=False)

        print(f"Total reps in comments file: {final_comments_df['rep'].nunique()}")
    else:
        print("No new comments to save.")